# 37 - Full-cohort tapered P&P, worker 0/2

This is shard 0 of 2 over the exact corrected 1,300-episode LIBERO-PRO cohort: 13 suites x 10 tasks x init indices 0-9. It collects only the always-on tapered-refinement intervention, so each worker runs 650 rollouts.

The source model executes 10 actions per generated 50-action chunk. At K=5 Euler steps `(3,4)`, refine-last feedback has weight 1 on action positions 0-9, decays linearly over positions 10-19, and is zero over positions 20-49.

Every 25 new rollouts, the table prints the current tapered SR beside two exact full-cohort references: **historical full PnP** and **historical unrefined**. Both references are selected by the corrected 10-action config hashes, so old 20-action rows cannot enter them. Worker 0 may set `EPISODE_LIMIT = 1` for a smoke test, then restore `None`; that row resumes safely.

## 1. Setup a fresh GPU runtime

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and resumable collection

In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.config import PI05_REPO_ID
from pnp.diversity import (SOURCE_TAPERED_FULL_EXPERIMENT,
    load_bootstrap_manifest, run_source_tapered_full_worker)

drive.mount('/content/drive')

EPISODES_PER_TASK = 10
SHARD_COUNT = 2
SHARD_INDEX = 0
EPISODE_LIMIT = None  # optional smoke test: set 1 once, then restore None
EXPERIMENT = SOURCE_TAPERED_FULL_EXPERIMENT
MANIFEST_PATH = Path(
    '/content/drive/MyDrive/pnp_diversity_v2/bootstrap_manifest_finetuned_v2.json')
manifest = load_bootstrap_manifest(MANIFEST_PATH)
assert manifest['source_model'] == PI05_REPO_ID, manifest['source_model']
SOURCE_MODEL_REVISION = manifest['source_model_revision']
assert SOURCE_MODEL_REVISION, 'v2 manifest is missing source_model_revision'

print({'experiment': EXPERIMENT, 'intervention': 'always-on tapered PnP',
       'n_action_steps': 10, 'generated_chunk_size': 50,
       'episodes_per_task': EPISODES_PER_TASK,
       'full_cohort_identities': 1300, 'identities_in_this_shard': 650,
       'shard_count': SHARD_COUNT, 'shard_index': SHARD_INDEX,
       'episode_limit': EPISODE_LIMIT,
       'progress_references': ['historical full PnP', 'historical unrefined'],
       'manifest_hash': manifest['manifest_hash'],
       'source_model_revision': SOURCE_MODEL_REVISION})
run_source_tapered_full_worker(
    episodes_per_task=EPISODES_PER_TASK, episode_limit=EPISODE_LIMIT,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    manifest_hash=manifest['manifest_hash'],
    source_model_revision=SOURCE_MODEL_REVISION, experiment=EXPERIMENT)